---
layout: post
title: "Python Libraries Homework"
description: "Summarize personal fencing bouts and count down to the next bout using four standard library modules."
date: 2026-10-04
categories: [HWHacks]
permalink: /hwhacks/2026-10-04-python-libraries/
hide: false
codemirror: true
---

## My Fencing Season Summary

Use **Run** (or **Ctrl+Enter / Cmd+Enter**) to summarize your fencing season in the browser. Your results are September 12, 2026: **15–12** against **Opponent 12**, and September 19, 2026: **15–9** against **Opponent 9**. The next competition is **October 23, 2026**. Each bout stores its date, opponent label, your touches, and the opponent’s touches.

`summarize_season(bouts)` returns total touches, average touches, the highest-scoring bout, and touches against each opponent. The docstring documents its inputs, outputs, empty-season behavior, and tie handling.

The runner uses four standard library modules: `json` to load and display records, `statistics` to find the mean, `math` to round a practice target up to a whole touch, and `datetime` to calculate days until your next bout. The summary totals **30 touches**, averages **15.0 touches** per bout, and sets a whole-number practice target of **15 touches**. Both bouts have 15 touches, so the first bout is selected as the best bout under the documented tie rule. The competition countdown updates using the current date when you run the code.

The imagined fencing app uses `pandas` for a season table and `Flask` for its scoreboard page. [Download requirements.txt]({{ site.baseurl }}/assets/python-libraries-app/requirements.txt). Standard library modules need no separate installation.


In [ ]:
# CODE_RUNNER: Summarize your fencing results and count down to your October 23 competition. | python_runtime: browser
import json
import math
import statistics
from datetime import date

# Your scores are recorded as touches scored and touches conceded.
# The supplied opponent numbers are used as labels for these two bout records.
season_json = """[
    {"date": "2026-09-12", "opponent": "Opponent 12", "touches": 15, "touches_against": 12},
    {"date": "2026-09-19", "opponent": "Opponent 9", "touches": 15, "touches_against": 9}
]"""
bouts = json.loads(season_json)

# Your next fencing competition is October 23, 2026.
next_bout = date(2026, 10, 23)


def summarize_season(season):
    """Return scoring statistics for a fencing season.

    Parameters:
        season (list[dict]): Bout records containing date (ISO YYYY-MM-DD
            string), opponent (string), and touches (nonnegative integer
            scored by you). An optional touches_against field records the
            opponent's score and is preserved in best_bout.

    Returns:
        dict: total_touches is the sum of all scores; average_touches is the
            mean rounded to one decimal place; best_bout is a copy of the
            highest-scoring bout record; touches_by_opponent maps each
            opponent's name to the combined touches scored against them.
            The first highest-scoring bout wins a tie. An empty season
            returns 0 total touches, 0.0 average touches, None for best_bout,
            and an empty touches_by_opponent dictionary.
    """
    touches = [bout["touches"] for bout in season]
    touches_by_opponent = {}
    for bout in season:
        opponent = bout["opponent"]
        touches_by_opponent[opponent] = touches_by_opponent.get(opponent, 0) + bout["touches"]

    best_bout = max(season, key=lambda bout: bout["touches"], default=None)
    return {
        "total_touches": sum(touches),
        "average_touches": round(statistics.mean(touches), 1) if touches else 0.0,
        "best_bout": dict(best_bout) if best_bout is not None else None,
        "touches_by_opponent": touches_by_opponent,
    }


report = summarize_season(bouts)
report["sport"] = "Fencing"
# ceil rounds up so a practice target is a whole number of touches.
report["practice_target_touches"] = math.ceil(report["average_touches"]) if bouts else None

# Subtracting dates produces a timedelta; .days gives the countdown in days.
days_left = (next_bout - date.today()).days if next_bout is not None else None
report["next_bout_date"] = next_bout.isoformat() if next_bout is not None else None
report["days_until_next_bout"] = days_left

print(json.dumps(report, indent=2))
if not bouts:
    print("Add your actual fencing bout results to season_json to summarize your season.")
if days_left is None:
    print("Set next_bout to your actual next bout or competition date for the countdown.")
elif days_left > 0:
    print(f"Next fencing bout or competition is in {days_left} days.")
elif days_left == 0:
    print("Next fencing bout or competition is today!")
else:
    print(f"That date was {-days_left} days ago. Set your next_bout date.")
